# Graph Compilation


BrainTrace compiles a recurrent model into an `ETraceGraph`, the intermediate representation connecting ETP parameters to hidden states. This page starts from one small RNN and then calls `compile_etrace_graph` directly.


## Single-Layer RNN

We start with the simplest case: a single recurrent layer followed by a linear readout.
The `ValinaRNNCell` contains one hidden state and one recurrent weight, and the `Linear`
readout has its own weight that feeds into the output.

In [1]:
import jax.numpy as jnp
import brainstate
import braintrace

In [2]:
class SingleLayerRNN(brainstate.nn.Module):
    def __init__(self, n_in, n_rec, n_out):
        super().__init__()
        self.rnn = braintrace.nn.ValinaRNNCell(n_in, n_rec)
        self.out = braintrace.nn.Linear(n_rec, n_out)

    def update(self, x):
        return self.out(self.rnn(x))


model = SingleLayerRNN(10, 32, 5)

# braintrace.compile initialises states, compiles the ETP graph, and returns a ready learner.
# We compile for a single unbatched sample (no batch_size), so the hidden state is (32,) and
# the recurrent op is the matrix-vector primitive etp_mv. verbose=2 prints full diagnostics.
learner = braintrace.compile(model, braintrace.D_RTRL, jnp.zeros(10), verbose=2)
learner.show_graph()

The hidden groups are:

   Group 0: [('rnn', 'h')]


The weight parameters which are associated with the hidden states are:

   Weight 0: ('rnn', 'W', 'weight')  is associated with hidden group 0


The non-etrace weight parameters are:

   Weight 0: ('out', 'weight')  (excluded: relation_excluded_non_temporal)


Compiler diagnostics (warnings / errors):

   [warning] relation_excluded_non_temporal: ETP primitive etp_mv (weight=('out', 'weight')) has no connected hidden states. It will be treated as a non-temporal parameter.



The hidden groups are:

   Group 0: [('rnn', 'h')]


The weight parameters which are associated with the hidden states are:

   Weight 0: ('rnn', 'W', 'weight')  is associated with hidden group 0


The non-etrace weight parameters are:

   Weight 0: ('out', 'weight')  (excluded: relation_excluded_non_temporal)





D:\BrainTrace\.venv\Lib\site-packages\braintrace\_compiler\hid_param_op.py:969: UserWarning: ETP primitive etp_mv (weight=('out', 'weight')) has no connected hidden states. It will be treated as a non-temporal parameter.
  _emit_no_relation_diag(


The output shows:

- **Hidden Group 0**: the hidden state of the `ValinaRNNCell` (path `('rnn', 'h')`)
- **Associated weight**: the recurrent weight inside the RNN cell (`('rnn', 'W', 'weight')`),
  eligibility-traced because it feeds the hidden state through an ETP primitive
- **Non-etrace weight**: the readout weight (`('out', 'weight')`). `braintrace.nn.Linear`
  *does* use an ETP primitive, but the readout's output is the network's final output and
  never flows back into a hidden state -- so the compiler reports it as a non-temporal
  parameter (still trained, just not through an eligibility trace). A matching
  `has no connected hidden states` warning is emitted at compile time.

This tells us that `D_RTRL` will maintain an eligibility trace for the recurrent weight,
tracking how it influences the hidden state over time.

## Using `compile_etrace_graph` Directly

Most users should call `braintrace.compile(...)` — it initialises states, compiles the graph, and returns a learner in one call (access the underlying graph via `learner.graph`). For advanced users who want to inspect the graph *without* wrapping the model in an algorithm, `braintrace.compile_etrace_graph()` is also available. This is useful for:

- Debugging model structure before training
- Verifying that ETP primitives are correctly placed
- Building custom online learning algorithms on top of the graph

In [3]:
model_direct = SingleLayerRNN(10, 32, 5)
brainstate.nn.init_all_states(model_direct)

graph_direct = braintrace.compile_etrace_graph(model_direct, jnp.zeros(10))

print(f"Number of hidden groups: {len(graph_direct.hidden_groups)}")
print(f"Number of relations: {len(graph_direct.hidden_param_op_relations)}")
print(f"Has perturbation: {graph_direct.hidden_perturb is not None}")

print("\nGraph fields:")
for key in graph_direct.dict().keys():
    print(f"  {key}")

Number of hidden groups: 1
Number of relations: 1
Has perturbation: True

Graph fields:
  module_info
  hidden_groups
  hid_path_to_group
  hidden_param_op_relations
  hidden_perturb
  diagnostics


The `compile_etrace_graph()` function returns the same `ETraceGraph` named tuple that is
stored internally by `D_RTRL` and other algorithms. You can use it to build custom
training loops or to programmatically analyze model structure.

## Summary

Graph compilation discovers the parameter-to-hidden and hidden-to-hidden relations used by an online-learning algorithm. Continue with [Visualization](visualization.ipynb) to inspect reports and compare multi-layer or convolutional graphs.
